# Susanoo Solar Wind plot -- 2026/3/30

In [ ]:
import matplotlib.pyplot as plt
import os
import sys
sys.path.append("./lib/")
import susanoo_lib as susanoo

# parameter set -- please modify !

In [ ]:
"""
[Susanoo QLツール] --- data download機能付
<所在>  https://drive.google.com/drive/folders/1FjRElo97g3IiT69K6kRDPI-11EMZfI2a
<出力>  "./plot" にあり
<元データ(CDF)> "./data/" にあり。

**** [注意] CDF fileの「version」を決め打ちしているので、あちらが変われば変更要。
****       自動的に最新データを読んでplotするように改装もできるが、やっていない。
****       現在は以下で固定： <./lib/susanoo_lib.py>  "name_solarwind_data"
****                         L.14    s_ver = 'v01.01'

<元設定>
Epoch_min     = '2025-08-01'  # Start time    変更可
Epoch_max     = '2025-08-31'  # End time      変更可
str_object    = 'mercury'     # Target object: 'mercury'       変更可
mode_download = 1             # 以下から./data/へdownload   https://chs.isee.nagoya-u.ac.jp/data/chs/simulation/susanoo/data/cdf/
                              #    "0"   no download。./data/ 下のCDFデータを読む
mode_plot     = 1             # ./plot/ の下に図を生成
                              #    "0"   生成しない。

<補足：QL>   https://chs.isee.nagoya-u.ac.jp/data/chs/simulation/susanoo/data/img/ql/mercury/  --> (yyyy)　に一月単位のQLがある。
"""

In [ ]:
Epoch_min  = '2025-08-01'               # Start
Epoch_max  = '2025-08-31'               # End   
str_object = 'mercury'                  # Target object: 'mercury'

In [ ]:
mode_download = 1                       # [download]    0: none     1: download from https://chs.isee.nagoya-u.ac.jp/data/chs/simulation/susanoo/data/cdf/
mode_plot     = 1                       # [plot]        0: none     1: plot dump

# *** Directory set ***
data_dir  = './data/data-Susanoo/'      # CDF Data folder
plot_dir  = './plot/'                   # Plot dump folder
if os.path.isdir('./data') == False:  os.mkdir('./data')
if os.path.isdir(data_dir) == False:  os.mkdir(data_dir)
if os.path.isdir(plot_dir) == False:  os.mkdir(plot_dir)

In [ ]:
import datetime
Epoch = datetime.datetime.strptime(Epoch_min[0:10], "%Y-%m-%d")     # Start epoch
Epoch = Epoch.strftime('%Y-%m-%d ')
print('Epoch:', Epoch)
Epoch_YYYY = Epoch[0:4];  Epoch_MM = Epoch[5:7];  Epoch_DD = Epoch[8:10]

######################
# download necessary kernels using wget (for Linux&WSL2) or curl -O (for Mac)
d_CDF = data_dir + '/' + str_object + '/' + Epoch_YYYY + '/' + Epoch_MM
s_CDF = 'https://chs.isee.nagoya-u.ac.jp/data/chs/simulation/susanoo/data/cdf/' + str_object + '/' + Epoch_YYYY + '/' + Epoch_MM + '/susanoo_sw_' + str_object + '_5m_' + Epoch_YYYY + Epoch_MM + Epoch_DD + '_v01.01.cdf'
print('CDF directory:', d_CDF)
print('CDF file:', s_CDF)
!wget -N -P $d_CDF $s_CDF
######################

# Data Read

In [ ]:
solarwind  = susanoo.read(Epoch_min, Epoch_max, data_dir, str_object, mode_download)

In [ ]:
if solarwind.num == 0:
    print("!!!!! Solar Wind: NO DATA !!!!!")
else:
    print('[Total]', solarwind.num, '     (', solarwind.epoch[0], '-', solarwind.epoch[-1],  ')')

# Plot

In [ ]:
n0 = 0;                     
n1 = -1

fig = plt.figure(figsize=(14, 11))
ax1 = fig.add_subplot(4, 1, 1);  ax2 = fig.add_subplot(4, 1, 2);  ax3 = fig.add_subplot(4, 1, 3);  ax4 = fig.add_subplot(4, 1, 4)
ax1.plot(solarwind.epoch[n0:n1], solarwind.dens[n0:n1],   '-b', ms=1, linewidth=1.0)
ax2.plot(solarwind.epoch[n0:n1], solarwind.pre [n0:n1],   '-b', ms=1, linewidth=1.0)
ax3.plot(solarwind.epoch[n0:n1], (solarwind.swvv[n0:n1,0]**2+solarwind.swvv[n0:n1,1]**2+solarwind.swvv[n0:n1,2]**2)**0.5, '-k', ms=1, linewidth=2.0, label="total")
ax3.plot(solarwind.epoch[n0:n1], solarwind.swvv[n0:n1,0], '-r', ms=1, linewidth=1.0, label="X")
ax3.plot(solarwind.epoch[n0:n1], solarwind.swvv[n0:n1,1], '-g', ms=1, linewidth=1.0, label="Y")
ax3.plot(solarwind.epoch[n0:n1], solarwind.swvv[n0:n1,2], '-b', ms=1, linewidth=1.0, label="Z")
ax4.plot(solarwind.epoch[n0:n1], (solarwind.imfb[n0:n1,0]**2+solarwind.imfb[n0:n1,1]**2+solarwind.imfb[n0:n1,2]**2)**0.5, '-k', ms=1, linewidth=2.0, label="total")
ax4.plot(solarwind.epoch[n0:n1], solarwind.imfb[n0:n1,0], '-r', ms=1, linewidth=1.0, label="X")
ax4.plot(solarwind.epoch[n0:n1], solarwind.imfb[n0:n1,1], '-g', ms=1, linewidth=1.0, label="Y")
ax4.plot(solarwind.epoch[n0:n1], solarwind.imfb[n0:n1,2], '-b', ms=1, linewidth=1.0, label="Z")
ax1.set_ylabel("Density (/cc)");  ax2.set_ylabel("Pressure (dyn/cm2)");  ax3.set_ylabel("Velocity (km/s)");  ax4.set_ylabel("IMF-B (nT)");  
title_date = "[Susanoo Solar Wind @ " + str_object + "]  " + solarwind.epoch[n0].strftime('%Y%m%d%H%M-') + solarwind.epoch[n1].strftime('%Y%m%d%H%M');  ax1.set_title(title_date)
ax3.legend(loc='upper right', fontsize=8);  ax4.legend(loc='upper right', fontsize=8)

xlim=[solarwind.epoch[n0], solarwind.epoch[n1]];  ax1.set_xlim(xlim); ax2.set_xlim(xlim); ax3.set_xlim(xlim); ax4.set_xlim(xlim)

fig.subplots_adjust(hspace=0.2);  fig.show
if mode_plot == 1:
    png_filename = plot_dir + 'SW-' + str_object + '_' + solarwind.epoch[n0].strftime('%Y%m%d%H%M-') + solarwind.epoch[n1].strftime('%Y%m%d%H%M') + '.png'
    fig.savefig(png_filename);  print(png_filename)